In [ ]:
from pathlib import Path
from collections import OrderedDict

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers, models
from tensorflow.keras.utils import image_dataset_from_directory

# ====================================================
# 1. Global configuration
# ====================================================
IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 123

root_dir = Path(
    "/data/users/zhouz6436/Pneumonia_image_classification/chest_xray_3class"
)
train_dir = root_dir / "train"
test_dir = root_dir / "test"
model_dir = root_dir.parent / "models"
model_dir.mkdir(parents=True, exist_ok=True)

AUTOTUNE = tf.data.AUTOTUNE

tf.keras.utils.set_random_seed(SEED)


In [ ]:
# ====================================================
# 2. Dataset preparation
# Keep images in the original 0-255 range; MobileNetV2 scaling
# is applied inside the model after data augmentation.
# ====================================================
def preprocess(image, label):
    return tf.cast(image, tf.float32), label


def prepare(dataset, shuffle=False):
    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=1000,
            seed=SEED,
            reshuffle_each_iteration=True,
        )

    return (
        dataset
        .map(preprocess, num_parallel_calls=AUTOTUNE)
        .batch(BATCH_SIZE)
        .prefetch(AUTOTUNE)
    )


In [ ]:
# ====================================================
# 3. Fixed 90/10 training-validation split and test set
# ====================================================
train_raw = image_dataset_from_directory(
    train_dir,
    validation_split=0.10,
    subset="training",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    shuffle=True,
    batch_size=None,
)

val_raw = image_dataset_from_directory(
    train_dir,
    validation_split=0.10,
    subset="validation",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    shuffle=True,
    batch_size=None,
)

test_raw = image_dataset_from_directory(
    test_dir,
    image_size=(IMG_SIZE, IMG_SIZE),
    shuffle=False,
    batch_size=None,
)

class_names = train_raw.class_names
num_classes = len(class_names)
expected_classes = ["NORMAL", "bacteria", "virus"]

assert class_names == expected_classes, (
    f"Expected {expected_classes}, found {class_names}"
)
assert val_raw.class_names == class_names
assert test_raw.class_names == class_names

print("Classes:", class_names)
print("Training images:", train_raw.cardinality().numpy())
print("Validation images:", val_raw.cardinality().numpy())
print("Test images:", test_raw.cardinality().numpy())


In [ ]:
# ====================================================
# 4. Nested training subsets and class weights
# ====================================================
total_train = train_raw.cardinality().numpy()


def make_raw_subset(fraction):
    number_of_images = max(1, int(total_train * fraction))
    subset = train_raw.take(number_of_images)
    print(f"{int(fraction * 100)}% subset: {number_of_images} images")
    return subset


def calculate_class_weights(raw_dataset):
    labels = np.array([
        int(label.numpy())
        for _, label in raw_dataset
    ])
    present_classes = np.unique(labels)

    assert len(present_classes) == num_classes, (
        f"Subset is missing a class: found labels {present_classes}"
    )

    weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(num_classes),
        y=labels,
    )

    return {
        class_index: float(weight)
        for class_index, weight in enumerate(weights)
    }


raw_subsets = OrderedDict({
    "10%": make_raw_subset(0.10),
    "25%": make_raw_subset(0.25),
    "50%": make_raw_subset(0.50),
    "100%": train_raw,
})

training_subsets = OrderedDict({
    name: prepare(raw_dataset, shuffle=True)
    for name, raw_dataset in raw_subsets.items()
})

subset_class_weights = {
    name: calculate_class_weights(raw_dataset)
    for name, raw_dataset in raw_subsets.items()
}

val = prepare(val_raw, shuffle=False)
test = prepare(test_raw, shuffle=False)

for name, weights in subset_class_weights.items():
    print(name, "class weights:", weights)


In [ ]:
# ====================================================
# 5. Model B: frozen ImageNet MobileNetV2 feature extractor
# ====================================================
data_augmentation = tf.keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.10),
    ],
    name="data_augmentation",
)


def make_classifier_B():
    backbone = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights="imagenet",
        pooling="avg",
    )
    backbone.trainable = False

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = data_augmentation(inputs)
    x = layers.Rescaling(1.0 / 127.5, offset=-1)(x)
    x = backbone(x, training=False)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


model_B = make_classifier_B()
model_B.summary()

assert model_B.output_shape[-1] == 3


In [ ]:
# ====================================================
# 6. Train and save one Model B for each data fraction
# ====================================================
EPOCHS = 30

histories_B = {}
tests_B = {}
models_B = OrderedDict()

for name, training_dataset in training_subsets.items():
    print(f"\n=== Model B three-class training: {name} subset ===")

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)

    model = make_classifier_B()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.20,
            patience=2,
            min_lr=1e-6,
            verbose=1,
        ),
    ]

    history = model.fit(
        training_dataset,
        epochs=EPOCHS,
        validation_data=val,
        class_weight=subset_class_weights[name],
        callbacks=callbacks,
        verbose=1,
    )

    histories_B[name] = history.history
    tests_B[name] = model.evaluate(test, return_dict=True, verbose=1)
    models_B[name] = model

    fraction_label = name.replace("%", "pct")
    weights_path = (
        model_dir
        / f"model_B_3class_{fraction_label}.weights.h5"
    )
    model.save_weights(weights_path)
    print("Saved:", weights_path)

print("\nModel B three-class test metrics:")
for name, metrics in tests_B.items():
    print(name, metrics)


In [ ]:
# ====================================================
# 7. Plot training histories
# ====================================================
class HistoryWrapper:
    def __init__(self, history_dictionary):
        self.history = history_dictionary


def plot_history(history, title_prefix):
    accuracy = history.history.get("accuracy", [])
    validation_accuracy = history.history.get("val_accuracy", [])
    loss = history.history.get("loss", [])
    validation_loss = history.history.get("val_loss", [])

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(accuracy, label="training accuracy")
    plt.plot(validation_accuracy, label="validation accuracy")
    plt.title(f"{title_prefix} accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(loss, label="training loss")
    plt.plot(validation_loss, label="validation loss")
    plt.title(f"{title_prefix} loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.tight_layout()
    plt.show()


for fraction, history_dictionary in histories_B.items():
    plot_history(
        HistoryWrapper(history_dictionary),
        title_prefix=f"Model B three-class ({fraction})",
    )


In [ ]:
# ====================================================
# 8. Three-class test evaluation
# ====================================================
y_true = np.concatenate([
    labels.numpy().ravel()
    for _, labels in test_raw
])

label_indices = list(range(num_classes))

print("Class mapping:")
for index, class_name in enumerate(class_names):
    print(index, "=", class_name)

for name, model in models_B.items():
    print(f"\n=== Model B evaluation: {name} subset ===")

    probabilities = model.predict(test, verbose=1)
    y_pred = np.argmax(probabilities, axis=1)

    print("Predicted-label counts:")
    for index, class_name in enumerate(class_names):
        print(class_name, ":", int(np.sum(y_pred == index)))

    print("\nClassification report:")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=label_indices,
            target_names=class_names,
            digits=4,
            zero_division=0,
        )
    )

    matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=label_indices,
    )
    print("Confusion matrix:")
    print(matrix)

    balanced_accuracy = balanced_accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    print("Balanced accuracy:", round(balanced_accuracy, 4))
    print("Macro F1:", round(macro_f1, 4))


# Model B three-class experiment

This notebook trains a frozen ImageNet-pretrained MobileNetV2 feature extractor
to distinguish **NORMAL**, **bacterial pneumonia**, and **viral pneumonia**.

Before running, create `chest_xray_3class/` using the data-preparation step in
`Model A-3Classifier.ipynb`. Restart the kernel, run all code cells in order,
and verify that TensorFlow reports exactly three classes. The notebook saves one
set of Model B weights per training fraction for use by Model C.
